# MCP Setup Repository Scaffolding

This notebook will help scaffold the MCP setup repository with modular Python files. We'll create:

1. A modular structure under `mcp_setup/` directory
2. A top-level `setup_mcp.py` script as the entry point

## Generate MCP Setup Modules

We'll create the following Python modules:
- `mcp_setup/configurator.py`: Handles server configurations and environment loading
- `mcp_setup/validator.py`: Validates environment variables and files
- `mcp_setup/generator.py`: Builds and writes the `.vscode/mcp.json` configuration
- `setup_mcp.py`: CLI entry point that chains the other modules

## Write `configurator.py`

This module will handle server configurations and environment loading logic.

In [178]:
%%writefile mcp_setup/configurator.py
"""
MCP Setup Configurator Module

This module handles server configurations and environment loading logic for MCP setup.
"""
import os
import json
from pathlib import Path
from typing import Dict, List, Optional, Any


class Configurator:
    """
    Handles configuration loading and management for MCP setup.
    """
    def __init__(self, env_file: str = ".env"):
        """
        Initialize the configurator with an environment file.

        Args:
            env_file: Path to the environment file (.env)
        """
        self.env_file = env_file
        self.env_vars = {}
        self.config = {}

    def load_environment(self) -> Dict[str, str]:
        """
        Load env vars from self.env_file, populate os.environ, and return a dict.
        Handles:
          • UTF-8 files on any platform
          • inline comments after a '#'
        Prints a simple ASCII debug line for each key it loads so Windows shells
        don't choke on Unicode.
        """
        env_vars: Dict[str, str] = {}
        if not os.path.exists(self.env_file):
            print(f"[WARN] Environment file {self.env_file} not found.")
            return env_vars

        with open(self.env_file, "r", encoding="utf-8", errors="ignore") as fh:
            for raw in fh:
                line = raw.strip()
                if not line or line.startswith("#"):
                    continue
                if "=" not in line:
                    continue

                key, val_raw = line.split("=", 1)
                key = key.strip()
                # strip the part after a '#' comment – works even if none present
                val = val_raw.split("#", 1)[0].strip().strip("'\"")
                env_vars[key] = val
                os.environ[key] = val
                print(f"[DBG] Loaded {key}={val}")

        self.env_vars = env_vars
        return env_vars



    def get_server_config(self, server_type: str) -> Dict[str, Any]:
        """
        Get server configuration based on server type.

        Args:
            server_type: Type of server (e.g., "production", "development")

        Returns:
            Server configuration dictionary
        """
        # Default configurations based on server type
        configs = {
            "production": {
                "debug": False,
                "host": self.env_vars.get("PROD_HOST", "localhost"),
                "port": int(self.env_vars.get("PROD_PORT", "8080")),
                "workers": 4
            },
            "development": {
                "debug": True,
                "host": self.env_vars.get("DEV_HOST", "localhost"),
                "port": int(self.env_vars.get("DEV_PORT", "8000")),
                "workers": 1
            },
            "testing": {
                "debug": True,
                "host": "localhost",
                "port": 8888,
                "workers": 1
            }
        }

        return configs.get(server_type, configs["development"])

    def load_json_config(self, config_file: str) -> Dict[str, Any]:
        """
        Load configuration from a JSON file.

        Args:
            config_file: Path to the JSON configuration file

        Returns:
            Configuration dictionary
        """
        if not os.path.exists(config_file):
            print(f"Warning: Configuration file {config_file} not found.")
            return {}

        with open(config_file, "r") as f:
            config = json.load(f)

        self.config = config
        return config

    def get_config_value(self, key: str, default: Any = None) -> Any:
        """
        Get a configuration value by key.

        Args:
            key: Configuration key
            default: Default value if key is not found

        Returns:
            Configuration value or default
        """
        return self.config.get(key, default)


if __name__ == "__main__":
    # Example usage
    configurator = Configurator()
    env_vars = configurator.load_environment()
    print("Environment Variables:", env_vars)

    server_config = configurator.get_server_config("development")
    print("Server Configuration:", server_config)


Overwriting mcp_setup/configurator.py


## Write `validator.py`

This module will handle validation of environment variables and files.

In [179]:
%%writefile mcp_setup/validator.py
# mcp_setup/validator.py
"""
MCP Setup Validator Module

This module validates environment variables and files required for MCP setup.
"""
import os
import pathlib
from typing import Dict, List, Optional, Any
from mcp_setup.configurator import Configurator
from importlib.resources import files

# ──────────────────────────────────────────────────────────────────────────────
# Compute the code executor path in a way that works for both development and installed environments
def _code_executor_path() -> pathlib.Path:
    # First, try the workspace directory (development mode)
    root = pathlib.Path(__file__).resolve().parent.parent
    workspace_path = root / "mcp_code_executor" / "build" / "index.js"
    if (workspace_path.exists()):
        return workspace_path
    
    # Next, try the current working directory (for when running as a command)
    cwd_path = pathlib.Path.cwd() / "mcp_code_executor" / "build" / "index.js"
    if (cwd_path.exists()):
        return cwd_path
    
    # Finally, try the installed package path (for when installed as a package)
    try:
        return files("mcp_setup") / "mcp_code_executor" / "build" / "index.js"
    except ImportError:
        # Fall back to the original path and let it fail later if necessary
        return workspace_path

CODE_EXECUTOR_JS = _code_executor_path()
# ──────────────────────────────────────────────────────────────────────────────

# Define MCP servers with their required environment variables and files
MCP_SERVERS = {
    "Code Executor": {
        "type": "stdio",
        "command": [
            "node",
            str(CODE_EXECUTOR_JS)       # ← absolute path, but computed at runtime
        ],
        "env_vars": ["CONDA_ENV_NAME", "CODE_STORAGE_DIR"],
        "files": []
    },
    "Snowflake": {
        "type": "stdio",
        "command": ["npx", "mcp_snowflake_server"],
        "env_vars": [
            "SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER", "SNOWFLAKE_PASSWORD",
            "SNOWFLAKE_ROLE", "SNOWFLAKE_WAREHOUSE", "SNOWFLAKE_DATABASE",
            "SNOWFLAKE_SCHEMA"
        ],
        "files": []
    },
    "Jupyter": {
        "type": "stdio",
        "template": "jupyter-docker",
        "env_vars": ["JUPYTER_URL", "JUPYTER_TOKEN", "NOTEBOOK_PATH"],
        "files": []
    },
    "OracleDB": {
        "type": "stdio",
        "command": ["docker", "run", "-i", "--rm", "dmeppiel/oracle-mcp-server"],
        "env_vars": ["ORACLE_CONNECTION_STRING", "TARGET_SCHEMA"],
        "files": []
    },
    "nba_mcp Docs": {
        "type": "sse",
        "url": "https://gitmcp.io/ghadfield32/nba_mcp",
        "env_vars": [],
        "files": []
    },
    "Gitingest-MCP": {
        # Launch through uvx so we don’t rely on npx / global npm
        "type": "stdio",
        "command": [
            # the uvx absolute path is looked-up at runtime; see build_mcp_config
            "uvx",
            "--from", "git+https://github.com/puravparab/gitingest-mcp",
            "gitingest-mcp"
        ],
        "env_vars": [],
        "files": []
    }
}


class Validator:
    """
    Validates environment variables and files required for MCP setup.
    """
    def __init__(self, configurator: Optional[Configurator] = None):
        """
        Initialize the validator with a configurator.

        Args:
            configurator: Configurator instance
        """
        self.configurator = configurator or Configurator()

    def list_servers(self) -> List[str]:
        """List all available MCP servers."""
        return list(MCP_SERVERS.keys())

    def get_server_config(self, name: str) -> Dict[str, Any]:
        """Get the configuration for a specific server."""
        return MCP_SERVERS.get(name, {})

    def validate_server(self, name: str) -> Dict[str, Any]:
        """
        Validate environment variables and files for a server.
        Returns a dict: { valid: bool, missing_env: [...], missing_files: [...] }
        """
        if name not in MCP_SERVERS:
            return {
                "valid": False,
                "error": f"Unknown server: {name}",
                "missing_env": [],
                "missing_files": []
            }

        server_config = MCP_SERVERS[name]
        missing_env = [
            var for var in server_config.get("env_vars", [])
            if not os.environ.get(var)
        ]
        missing_files = [
            fp for fp in server_config.get("files", [])
            if not pathlib.Path(fp).is_file()
        ]

        return {
            "valid": not missing_env and not missing_files,
            "missing_env": missing_env,
            "missing_files": missing_files
        }

    def validate_servers(self, server_names: List[str]) -> Dict[str, Dict[str, Any]]: 
        """Validate multiple servers at once."""
        return {name: self.validate_server(name) for name in server_names}


if __name__ == "__main__":
    validator = Validator()
    print("Available servers:", validator.list_servers())
    print("Jupyter validation:", validator.validate_server("Jupyter"))



Overwriting mcp_setup/validator.py


## Write `generator.py`

This module will handle the generation and writing of the `.vscode/mcp.json` configuration file.

In [180]:
%%writefile mcp_setup/generator.py
"""
MCP Setup Generator Module

This module handles the generation and writing of the `.vscode/mcp.json` configuration file.
"""

import os
import json
import pathlib
import socket
import sys                        # ← to detect platform
from typing import Dict, List, Any, Optional

from mcp_setup.validator import Validator, MCP_SERVERS

class Generator:
    def __init__(self,
                 validator: Optional[Validator] = None,
                 env_file: str = ".env"):
        from pathlib import Path

        self.validator = validator or Validator()
        self.env_file  = env_file
        self.output_file = "mcp.json"

        # 1) Find the "workspace root" by looking for a .git folder:
        cwd = Path.cwd().resolve()
        for parent in [cwd] + list(cwd.parents):
            if (parent / ".git").exists():
                self.workspace_root = parent
                break
        else:
            self.workspace_root = cwd

        # 2) VS Code expects its .vscode folder at the workspace root
        self.output_dir = self.workspace_root / ".vscode"


    def build_mcp_config(self, server_names: List[str]) -> Dict[str, Any]:
        """
        Build .vscode/mcp.json, with OS-aware Jupyter networking:
        - Linux/macOS: port-map if free, else --network host
        - Windows: port-map if free, else drop port mapping
        """
        import shutil, socket, sys

        def is_port_free(port: int) -> bool:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                try:
                    s.bind(("", port))
                    return True
                except OSError:
                    return False

        cfg: Dict[str, Any] = {"servers": {}}
        on_windows = sys.platform.startswith("win")

        for name in server_names:
            scfg = MCP_SERVERS.get(name)
            if not scfg:
                print(f"[WARN] Unknown server '{name}' – skipped.")
                continue

            # --- Jupyter special case ---
            if scfg.get("template") == "jupyter-docker":
                missing = [v for v in scfg["env_vars"] if not os.getenv(v)]
                if missing:
                    print(f"[ERROR] Missing {missing} for Jupyter – skipping.")
                    continue

                url      = os.environ["JUPYTER_URL"]
                token    = os.environ["JUPYTER_TOKEN"]
                notebook = os.environ["NOTEBOOK_PATH"]

                docker_args = ["run", "-i", "--rm"]

                # Linux/macOS host-network override
                if not on_windows and os.getenv("MCP_HOST_NET","").lower() == "true":
                    docker_args += ["--network", "host"]
                else:
                    # Decide whether to port-map
                    if is_port_free(8888):
                        docker_args += [
                            "-e", "DOCKER_DEFAULT_PLATFORM=linux/amd64",
                            "-p", "8888:8888"
                        ]
                    else:
                        if on_windows:
                            print(
                              "[WARN] Port 8888 busy on host; skipping `-p 8888:8888`. "
                              "Container will reach your Jupyter at host.docker.internal:8888."
                            )
                            # no port-map
                        else:
                            # Linux/mac fallback
                            print(
                              "[INFO] Port 8888 busy; using --network host for Jupyter."
                            )
                            docker_args += ["--network", "host"]

                # Always point container at host.docker.internal
                docker_args += [
                    "-e", f"SERVER_URL={url}",
                    "-e", f"TOKEN={token}",
                    "-e", f"NOTEBOOK_PATH={notebook}",
                    "datalayer/jupyter-mcp-server:latest"
                ]

                cfg["servers"][name] = {
                    "type":    "stdio",
                    "command": "docker",
                    "args":    docker_args
                }
                continue

            # --- Default stdio servers ---
            if scfg["type"] == "stdio":
                cmd = list(scfg["command"])
                if cmd[0] == "uvx" and shutil.which("uvx"):
                    cmd[0] = shutil.which("uvx")
                if cmd[0] == "npx" and not shutil.which("npx"):
                    if shutil.which("uvx"):
                        print(f"[INFO] Replacing missing npx with uvx for {name}")
                        cmd[0] = shutil.which("uvx")
                    else:
                        print(f"[ERROR] Neither 'npx' nor 'uvx' found – {name} will fail.")

                entry = {"type": "stdio", "command": cmd[0]}
                if len(cmd) > 1:
                    entry["args"] = cmd[1:]
                    
                # ─── locate the actual .env file under the workspace ───
                from pathlib import Path
                ws = Path(self.workspace_root)
                # try root first
                if (ws / self.env_file).exists():
                    rel_env = self.env_file
                else:
                    # fallback: search all subfolders once
                    found = next(ws.rglob(self.env_file), None)
                    if found:
                        rel_env = str(found.relative_to(ws))
                    else:
                        # gives up: warning and leave as-is
                        print(f"[WARN] Could not find {self.env_file} under {ws}; using top‐level reference.")
                        rel_env = self.env_file

                entry["envFile"] = f"${{workspaceFolder}}/{rel_env}"
                if scfg.get("env_vars"):
                    entry["env"] = {v: os.getenv(v, "") for v in scfg["env_vars"]}

                cfg["servers"][name] = entry
                continue

            # --- SSE servers ---
            if scfg["type"] == "sse":
                url = scfg.get("url") or os.getenv(scfg.get("url_env",""), "")
                if not url:
                    print(f"[WARN] No URL for '{name}' – skipped.")
                    continue
                entry = {"type": "sse", "url": url}
                if scfg.get("env_vars"):
                    entry["env"] = {v: os.getenv(v, "") for v in scfg["env_vars"]}
                cfg["servers"][name] = entry

        return cfg



    def write_mcp_config(self, config: Dict[str, Any]) -> str:
        """
        Write the MCP configuration to file.

        Args:
            config: MCP configuration dictionary

        Returns:
            Path to the written configuration file
        """
        # Create output directory if it doesn't exist
        os.makedirs(self.output_dir, exist_ok=True)
        output_path = os.path.join(self.output_dir, self.output_file)

        # Use utf-8 explicitly
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)

        return output_path

    def generate_mcp_config(self, server_names: List[str]) -> Dict[str, Any]:
        """
        Generate MCP configuration for selected servers and write to file.

        Args:
            server_names: List of server names

        Returns:
            Dictionary with generation result, including:
                - config: Generated configuration
                - output_path: Path to the written configuration file
                - success: Whether the generation was successful
        """
        try:
            # Build configuration
            config = self.build_mcp_config(server_names)

            # Write configuration to file
            output_path = self.write_mcp_config(config)

            return {
                "success": True,
                "config": config,
                "output_path": output_path
            }
        except Exception as e:
            return {
                "success": False,
                "error": str(e)
            }


if __name__ == "__main__":
    # Example usage
    generator = Generator()

    # Generate MCP configuration for selected servers
    server_names = ["Jupyter", "Code Executor"]
    result = generator.generate_mcp_config(server_names)

    if result["success"]:
        print(f"Configuration written to {result['output_path']}")
        print("Configuration:")
        print(json.dumps(result["config"], indent=2))
    else:
        print(f"Error generating configuration: {result['error']}")


Overwriting mcp_setup/generator.py


## Write `setup_mcp.py`

This file will be the main entry point for the MCP setup tool.

In [181]:
%%writefile setup_mcp.py
#!/usr/bin/env python3
"""
MCP Setup CLI

This module provides a command-line interface for setting up MCP servers in VS Code.
"""
import os
import sys
import argparse
import json
from typing import List, Dict, Any, Optional

from mcp_setup.configurator import Configurator
from mcp_setup.validator import Validator
from mcp_setup.generator import Generator
from mcp_setup.validator import MCP_SERVERS

import subprocess, time, requests, importlib.util
from urllib.parse import urlparse, urljoin


def select_servers(validator: Validator) -> List[str]:
    """
    Present an interactive checklist of available MCP servers.

    Args:
        validator: Validator instance

    Returns:
        List of selected server names
    """
    servers = validator.list_servers()

    print("\nAvailable MCP Servers:")
    for i, name in enumerate(servers, 1):
        print(f"  {i}. {name}")

    print("\nSelect servers to enable (comma-separated indices, e.g., '1,3,5'):")
    choice = input("Choice: ").strip()

    selected = []
    for idx in choice.split(","):
        try:
            idx = int(idx.strip()) - 1
            if 0 <= idx < len(servers):
                selected.append(servers[idx])
        except ValueError:
            pass

    return selected



# 1) Replacement for select_servers():
def select_and_expand_servers(validator: Validator) -> List[str]:
    """
    1. Present a numbered list of available MCP servers.
    2. Let the user pick (comma-separated).
    3. For each selected server, ask if they need multiple connections.
       - If yes: prompt for labels, e.g. "prod,analytics"
       - Expand "Snowflake" → ["Snowflake:prod","Snowflake:analytics"]
    Returns the final list of server identifiers.
    """
    servers = validator.list_servers()
    print("\nAvailable MCP Servers:")
    for i, name in enumerate(servers, 1):
        print(f"  {i}. {name}")
    choice = input("\nSelect servers (e.g. 1,3,5): ").strip()

    raw = []
    for idx in choice.split(","):
        try:
            n = int(idx) - 1
            if 0 <= n < len(servers):
                raw.append(servers[n])
        except:
            pass

    expanded = []
    for server in raw:
        ans = input(f"❓ Multiple connections for '{server}'? (y/N): ").strip().lower()
        if ans == "y":
            labels = input(f"Enter labels for '{server}' (comma-separated): ").strip()
            for lbl in [l.strip() for l in labels.split(",") if l.strip()]:
                expanded.append(f"{server}:{lbl}")
        else:
            expanded.append(server)
    return expanded


# 2) New helper to prompt for missing env vars:
def prompt_for_env_vars(server_ids: List[str], env_file: str):
    """
    For each 'Server[:instance]', prompt for any required env vars that
    are still missing from os.environ, then append them to the env file.
    """
    from mcp_setup.validator import MCP_SERVERS

    updates = []
    for sid in server_ids:
        base = sid.split(":",1)[0]
        required = MCP_SERVERS[base]["env_vars"]
        for var in required:
            if not os.environ.get(var):
                val = input(f"🔑 Enter value for {var} (for {base}): ").strip()
                updates.append((var, val))
                os.environ[var] = val
                print(f"✅ Set {var}={val}")
    if updates:
        with open(env_file, "a") as f:
            for k, v in updates:
                f.write(f"\n{k}={v}")
        print(f"💾 Appended {len(updates)} new values to {env_file}")


def main() -> int:
    """
    Main entry point for the MCP setup tool.

    Returns:
        Exit code (0 for success, non-zero for error)
    """

    parser = argparse.ArgumentParser(description="Setup MCP servers for VS Code")
    parser.add_argument(
        "--env",
        choices=["dev", "stage", "prod"],
        required=True,
        help="Environment to use (dev, stage, or prod)"
    )
    args = parser.parse_args()

    # Initialize components
    env_file = f".env.{args.env}"
    configurator = Configurator(env_file)
    validator = Validator(configurator)
    generator = Generator(validator, env_file=os.path.basename(env_file))

    # Step 1: Load environment
    print(f"\n[INFO] Loading environment from {env_file} ...")
    try:
        env_vars = configurator.load_environment()
        if not env_vars:
            print(f"[WARN] No environment variables found in {env_file}.")
    except Exception as e:
        print(f"[ERROR] Error loading environment: {e}")
        return 1

    # Step 2: Present server checklist
    selected_servers = select_servers(validator)
    if not selected_servers:
        print("[ERROR] No servers selected. Exiting.")
        return 1

    print(f"\n[INFO] Selected servers: {', '.join(selected_servers)}")

    # Step 3: Prompt for missing credentials
    print("\n[INFO] Prompting for any missing credentials...")
    prompt_for_env_vars(selected_servers, env_file)
    print("[INFO] Environment now contains:",
          ", ".join(k for k in MCP_SERVERS
                          .get(selected_servers[0].split(":")[0], {})
                          .get("env_vars", [])
                    if k in os.environ))

    # Step 4: Validate servers
    print("\n[INFO] Validating selected servers...")
    validation_results = validator.validate_servers(selected_servers)

    valid_servers = []
    invalid_servers = []

    for name, result in validation_results.items():
        if result["valid"]:
            valid_servers.append(name)
            print(f"[OK]   {name}: Valid")
        else:
            invalid_servers.append((name, result))
            print(f"[FAIL] {name}: Invalid")
            if result["missing_env"]:
                print(f"       Missing env vars: {', '.join(result['missing_env'])}")
            if result["missing_files"]:
                print(f"       Missing files:    {', '.join(result['missing_files'])}")

    # Step 5: Generate MCP configuration
    if valid_servers:
        print("\n[INFO] Generating MCP configuration...")
        result = generator.generate_mcp_config(valid_servers)

        if result["success"]:
            print(f"[OK] Configuration written to {result['output_path']}")
            cfg_keys = ", ".join(result["config"].get("servers", {}).keys())
            print(f"[INFO] Configured servers: {cfg_keys}")
        else:
            print(f"[ERROR] Error generating configuration: {result.get('error', 'Unknown')}")
            return 1
    else:
        print("[WARN] No valid servers found. Configuration not generated.")
        return 1

    # Step 6: Summary
    print("\n[SUMMARY]")
    print(f"  - Successfully configured: {len(valid_servers)} server(s)")
    print(f"  - Failed to configure:     {len(invalid_servers)} server(s)")
    if invalid_servers:
        print("  Details of failures:")
        for name, res in invalid_servers:
            print(f"    * {name}: missing {res.get('missing_env') or res.get('missing_files')}")

    print("\n[INFO] Done. You can verify in VS Code with 'MCP: List Servers'.")
    return 0



if __name__ == "__main__":
    sys.exit(main())


Overwriting setup_mcp.py


## Create `__init__.py` for the Package

Let's create an `__init__.py` file to make the `mcp_setup` directory a proper Python package.

In [182]:
%%writefile mcp_setup/__init__.py
"""
MCP Setup Package

This package provides tools for setting up MCP servers in VS Code.
"""

__version__ = "0.1.0"
__author__ = "MCP Setup Team"

from mcp_setup.configurator import Configurator
from mcp_setup.validator import Validator, MCP_SERVERS
from mcp_setup.generator import Generator

__all__ = ["Configurator", "Validator", "Generator", "MCP_SERVERS"]

Overwriting mcp_setup/__init__.py


## Create .gitignore File

Let's create a `.gitignore` file to exclude environment files and generated MCP configuration from version control.

In [183]:
%%writefile .gitignore
# Python
__pycache__/
*.py[cod]
*$py.class
*.so
.Python
build/
develop-eggs/
dist/
downloads/
eggs/
.eggs/
lib/
lib64/
parts/
sdist/
var/
wheels/
*.egg-info/
.installed.cfg
*.egg
MANIFEST

# Environments
.env
.env.dev
.env.stage
.env.prod
.venv
env/
venv/
ENV/
env.bak/
venv.bak/

# VSCode
.vscode/*
.vscode/mcp.json
!.vscode/settings.json
!.vscode/tasks.json
!.vscode/launch.json
!.vscode/extensions.json

# Jupyter Notebook
.ipynb_checkpoints

Overwriting .gitignore


## Create Sample .env.example File

Let's create a `.env.example` file as a template for environment variables.

In [184]:
%%writefile .env.example
# Sample environment variables for MCP servers

# Code Executor
CODE_STORAGE_DIR=code_executor_storage
CONDA_ENV_NAME=your_conda_env_name

# Snowflake
SNOWFLAKE_ACCOUNT=your_account
SNOWFLAKE_USER=your_user
SNOWFLAKE_PASSWORD=your_password
SNOWFLAKE_ROLE=your_role
SNOWFLAKE_WAREHOUSE=your_warehouse
SNOWFLAKE_DATABASE=your_database
SNOWFLAKE_SCHEMA=your_schema

# Jupyter
JUPYTER_URL=http://host.docker.internal:8888
JUPYTER_TOKEN=insert_token         # must match token used in jupyter lab command
NOTEBOOK_PATH=notebooks/demo.ipynb
# OracleDB
ORACLE_CONNECTION_STRING=username/password@//host:port/service
TARGET_SCHEMA=your_schema


Overwriting .env.example


# Tasks.py 


In [185]:
%%writefile tasks.py
"""
# tasks.py

This file contains Invoke tasks for automating various setup and development tasks.
"""
import os
import sys
import shutil
import subprocess    # ← new
import time          # ← new
from invoke import task, UnexpectedExit

# Cross-platform Python / venv paths
PYTHON = sys.executable
VENV = ".venv"
WIN = os.name == "nt"  # Added WIN constant for platform checks
if WIN:  # Windows
    VENV_PYTHON = os.path.join(VENV, "Scripts", "python.exe")
else:  # Unix/Linux/MacOS
    VENV_PYTHON = os.path.join(VENV, "bin", "python")

# Configuration
ENV_FILES = {
    "dev": ".env.dev",
    "stage": ".env.stage",
    "prod": ".env.prod",
}
NPM = "npm"
UVX = "@datalayer/uvx"
JUPYTER_PORT = "8888"

# 1. Bootstrap Python venv & install Python deps
@task
def setup_py(c, venv=".venv"):
    """
    Provision the Python environment with **uv**.

    1. If *venv* already exists, re-use it and just `uv sync`.
    2. If not, create it with `uv venv <path>`.
    3. Activate the venv and run `uv sync` to install all deps
       (project in editable-mode is automatic).
    """
    import shlex

    if os.path.isdir(venv):
        print(f"✅  Re-using existing venv: {venv}")
    else:
        c.run(f"uv venv {shlex.quote(venv)}", echo=True)

    # uv provides an activation script we can source inline for *nix shells;
    # on Windows Invoke will fall back to spawning a new cmd.exe run.
    activate = os.path.join(venv, "bin", "activate") if not WIN else None
    cmd_sync = f"uv sync --extra dev"  # include dev extras so lint/tests work

    if activate and os.path.isfile(activate):
        c.run(f". {activate} && {cmd_sync}", shell="/bin/bash", pty=not WIN, echo=True)
    else:  # Windows PowerShell / cmd
        vpy = os.path.join(venv, "Scripts", "python.exe")
        c.run(f"{vpy} -m uv sync --extra dev", echo=True)

# 2. Build the Code Executor sub-project
@task
def setup_js(c):
    c.run("cd mcp_code_executor && npm install && npm run build")

# 3. Install global UVX (if desired)
@task
def setup_uvx(c):
    """
    Attempt to install the (optional) uvx CLI.  If the package is missing
    from npm, print a warning but continue bootstrapping.
    """
    # 1) If uvx already on PATH, nothing to do
    found = shutil.which("uvx")
    if found:
        print(f"✅  `uvx` already installed at: {found}")
        return

    # 2) Attempt to install
    cmd = f"{NPM} install -g {UVX}"
    print(f"[DBG] Running: {cmd}")
    try:
        c.run(cmd, echo=True)
        print("✅  Successfully installed `uvx`.")
    except UnexpectedExit as e:
        stderr = e.result.stderr or ""
        print(f"[DBG] `npm` stderr:\n{stderr}")
        if "E404" in stderr:
            print("⚠️  Skipped uvx: package not found on npm (private?).")
        else:
            print("❌  Failed to install uvx (non-404 error), re-raising.")
            raise

# 4. Copy & prompt for your .env
@task
def init_env(c, env="dev"):
    """
    Copy .env.example → .env.{env} so you can fill in credentials.
    """
    src = ".env.example"
    dst = ENV_FILES.get(env)
    shutil.copy(src, dst)
    print(f"👉  Copied {src} → {dst}. Now edit {dst} with your values.")

# 5. Start JupyterLab (in host or container as needed)
@task
def jupyter(c, env="dev"):
    """
    Launch JupyterLab from the venv, using a clean token from .env.{env}.
    Cuts off any inline '# comment' so the CLI line is never truncated.
    """
    env_path = ENV_FILES.get(env)
    token = None
    if os.path.exists(env_path):
        with open(env_path, "r", encoding="utf-8", errors="ignore") as fh:
            for ln in fh:
                if ln.partition("=")[0].strip() == "JUPYTER_TOKEN":
                    token = ln.partition("=")[2].split("#", 1)[0].strip()
                    break

    cmd = (
        f"{VENV_PYTHON} -m jupyter lab "
        f"--port {JUPYTER_PORT} --ip=0.0.0.0 "
        + (f"--NotebookApp.token={token}" if token else "--NotebookApp.token=''")
    )
    c.run(cmd, pty=not WIN, echo=True)

# 6. Run the MCP setup and generate your VS Code config
@task
def mcp(c, env="dev"):
    """
    1) Start JupyterLab in the background (so the port and token are live).
    2) Wait a few seconds for it to spin up.
    3) Run setup_mcp.py --env {env} to configure VS Code.
    4) Leave Jupyter running so VS Code can actually connect.
    """
    env_path = ENV_FILES.get(env)
    token = None
    if os.path.exists(env_path):
        with open(env_path, "r", encoding="utf-8", errors="ignore") as fh:
            for ln in fh:
                if ln.partition("=")[0].strip() == "JUPYTER_TOKEN":
                    token = ln.partition("=")[2].split("#", 1)[0].strip()
                    break

    jupyter_cmd = [
        VENV_PYTHON, "-m", "jupyter", "lab",
        "--port", JUPYTER_PORT, "--ip=0.0.0.0",
    ]
    if token:
        jupyter_cmd.append(f"--NotebookApp.token={token}")
    else:
        jupyter_cmd.append("--NotebookApp.token=''")

    print(f"[INFO] Launching JupyterLab (token={token}) …")
    jproc = subprocess.Popen(jupyter_cmd)

    # give it a moment to bind to port 8888
    time.sleep(5)

    print(f"[INFO] Running MCP setup with env='{env}' …")
    c.run(f"{VENV_PYTHON} setup_mcp.py --env {env}", echo=True)

    print("[INFO] JupyterLab is still running in the background. ")
    print("       You can now open VS Code and do 'MCP: List Servers'.")
    # DO NOT terminate jproc here — leave it up!

# 7. A "meta" task to do it all (minus editing env)
@task(pre=[setup_py, setup_js, setup_uvx, init_env])
def bootstrap(c, env="dev"):
    """
    One-shot: Python deps (via **uv**), JS build, optional uvx, copy .env.

    After this finishes, edit ``.env.{env}`` with credentials
    and run `inv mcp --env={env}`.
    """
    # Normalize & validate the env parameter
    if not env:
        print("[WARN] No --env provided, defaulting to 'dev'")
        env = "dev"
    if env not in ("dev", "stage", "prod"):
        print(f"❌  Invalid env '{env}'. Must be one of dev, stage, prod.")
        sys.exit(1)

    print(f"🚀  Bootstrap complete with uv-managed environment for '{env}'!")
    print(f"👉  Next step: edit `.env.{env}` to fill in your credentials.")
    print(f"👉  Then run: `inv mcp --env={env}`")







Overwriting tasks.py


## Usage Instructions

Now that we have created all the necessary files, let's test the MCP setup tool.

### How to Use the MCP Setup Tool

1. **Create environment files**:
   Copy `.env.example` to `.env.dev` (or `.env.stage` or `.env.prod`) and fill in the required values.
   ```bash
   cp .env.example .env.dev
   # Edit .env.dev with your favorite text editor
   ```

2. **Run the setup script**:
   ```bash
   python setup_mcp.py --env dev
   ```

3. **Select servers** by entering their numbers (comma-separated).

4. **Verify the configuration** in VS Code:
   - Open VS Code in this workspace
   - Press `Ctrl+Shift+P` (or `Cmd+Shift+P` on Mac) to open the Command Palette
   - Type "MCP: List Servers" and press Enter
   - You should see your configured servers listed

5. **Troubleshooting**:
   - If a server is not configured, check the error message for missing environment variables or files.
   - Make sure the required environment variables are set in your `.env.dev` file.

## Testing the Setup

Let's create a sample `.env.dev` file for testing purposes. In a real scenario, you would manually create this file with your actual credentials.

In [186]:
%%writefile .env.dev
# Sample environment variables for MCP servers

# Code Executor
CODE_STORAGE_DIR=code_executor_storage
CONDA_ENV_NAME=your_conda_env_name

# Snowflake
SNOWFLAKE_ACCOUNT=your_account
SNOWFLAKE_USER=your_user
SNOWFLAKE_PASSWORD=your_password
SNOWFLAKE_ROLE=your_role
SNOWFLAKE_WAREHOUSE=your_warehouse
SNOWFLAKE_DATABASE=your_database
SNOWFLAKE_SCHEMA=your_schema

# Jupyter
JUPYTER_URL=http://host.docker.internal:8888
JUPYTER_TOKEN=abc12345         # must match token used in jupyter lab command
NOTEBOOK_PATH=notebooks/demo.ipynb

# OracleDB
ORACLE_CONNECTION_STRING=username/password@//host:port/service
TARGET_SCHEMA=your_schema


Overwriting .env.dev


Now you can test the MCP setup tool by running the following command in a terminal:

```bash
python setup_mcp.py --env dev
```

This will prompt you to select which servers to configure. Based on our sample `.env.dev` file, the following servers should be valid:

- Jupyter
- nba_mcp Docs

Other servers will be marked as invalid due to missing environment variables.

## Conclusion

We have successfully created a modular MCP setup tool that can:

1. Read environment variables from `.env.dev`, `.env.stage`, or `.env.prod` files
2. Present an interactive checklist of available MCP servers
3. Validate required environment variables and files for each selected server
4. Generate or update `.vscode/mcp.json` for the valid servers
5. Print a detailed report of successes and failures

This tool can be extended by adding more servers to the `MCP_SERVERS` dictionary in `validator.py`, or by adding more validation logic to check for specific requirements of each server.

## Next Steps

Here are some potential improvements for future versions:

1. **Add configuration options**: Allow users to customize server settings beyond the required environment variables.
2. **Add server health checks**: Verify that servers are running and accessible before configuring them.
3. **Add support for server dependencies**: Handle dependencies between servers (e.g., one server might require another to be running).
4. **Package as a CLI tool**: Convert this into a standalone CLI tool that can be installed via pip.
5. **Add automated tests**: Create unit tests to ensure the tool works as expected.
6. **Add error recovery**: Provide suggestions for fixing validation errors.

To package this as a CLI tool, you would need to create a `setup.py` file and add appropriate entry points. This would allow users to install the tool with `pip install .` and run it from anywhere with a simple command like `mcp-setup --env dev`.